In [11]:
from options_desk.deep_hedging import BuehlerTrainer, TrainerConfig, LiabilitySpec
from options_desk.deep_hedging.jax.env import DeepHedgingEnvConfig, FloatingOptionGrid
from options_desk.deep_hedging.jax.pricing import compile_padded_grid
from options_desk.deep_hedging.jax.pricing import HestonMarketParams

# grid = FloatingOptionGrid(maturities=(5, 10, 20, 40, 63, 126), moneyness_by_maturity={
#             5:  (0.97, 0.99, 1.00, 1.01, 1.03),
#             10: (0.95, 0.98, 1.00, 1.02, 1.05),                                             
#             20: (0.93, 0.97, 1.00, 1.03, 1.07),                                             
#             40: (0.90, 0.95, 1.00, 1.05, 1.10),                                             
#             63: (0.85, 0.93, 0.95, 1.00, 1.05, 1.07, 1.15), # 7 strikes, ±15%             
#             126: (0.70, 0.85, 0.95, 1.00, 1.05, 1.15, 1.30), # 7 strikes, ±30% 
# })



In [13]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

INITIAL_SPOT     = 1.0                                                                  
INITIAL_VARIANCE = 0.0625 
grid = FloatingOptionGrid(                                
      maturities=(10, 20, 40, 80, 120),                                                   
      moneyness_by_maturity={                                                             
          10:  (0.99, 1.00, 1.01),                                                        
          20:  (0.97, 0.99, 1.00, 1.01, 1.03),                                            
          40:  (0.95, 1.00, 1.05),                                                        
          80:  (0.91, 1.00, 1.09),                                                        
          120: (0.85, 0.95, 1.00, 1.05, 1.15),                                            
      },                                                                                  
  )      
env_cfg = DeepHedgingEnvConfig(
      horizon_steps=240, option_grid=grid, dt=1/250,                                      
      transaction_cost_underlying=1e-4,                                                   
      transaction_cost_option=1e-2,                                                       
  )                                                                                       
padded = compile_padded_grid(grid, dt=env_cfg.dt, horizon_steps=env_cfg.horizon_steps)  
liability = LiabilitySpec(                                                              
      kind='cliquet',                                                                     
      maturity=240,                                         
      cap=0.015,                                                                          
      floor=0.0,                                            
      quantity=-1.0,                                          # short cliquet             
      reset_dates=(20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240),               
  )       
train_cfg = TrainerConfig(                                                              
      policy_kind='lstm',                                                                 
      lstm_hidden_size=32,
      lstm_n_blocks=4,                                                                    
      lstm_position_limit=1.0,       # ← tighter (was 2.0)  
      lstm_last_layer_scale=1e-4,    # ← 10x smaller init (was 1e-3)                      
      risk_aversion=10.0,            # ← 100x smaller (was 1000)                          
      learning_rate=5e-5,            # ← 4x smaller (was 2e-4)                            
      grad_clip=0.5,                 # ← tighter (was 1.0)                                
      batch_size=512,                                                                     
      n_epochs=50,                   # ← shorter, just to check stability
      eval_every=5,                                                                       
  )                                                         
heston = HestonMarketParams(
      kappa=2.0,
      theta=0.04,
      sigma_v=0.3,
      rho=-0.7,
      r=0.0,
      q=0.0,
)

trainer = BuehlerTrainer.from_heston(                                                   
      config=train_cfg, env_config=env_cfg, market_params=heston,
      padded_grid=padded, liability=liability,                                            
      initial_spot=INITIAL_SPOT, initial_variance=INITIAL_VARIANCE,
      seed = 42,
) 
print(f"obs_dim={trainer.obs_dim}, n_instruments={trainer.n_instruments}, "             
        f"n_params={sum(p.numel() for p in trainer.policy.parameters()):,}")              
                                                                                          

obs_dim=120, n_instruments=39, n_params=39,079


In [14]:
history = trainer.train(progress=True)

train:   2%|▏         | 1/50 [00:01<01:21,  1.67s/it, loss=nan, var=nan, cost=0.0000, err_std=nan, err_mean=+nan]

Epoch  1/50  loss=nan  var=nan  cost=0.000001  hedging_err=nan +/- nan


train:  10%|█         | 5/50 [00:07<01:13,  1.63s/it, loss=nan, var=nan, cost=nan, err_std=nan, err_mean=+nan]   

Epoch  5/50  loss=nan  var=nan  cost=nan  hedging_err=nan +/- nan


train:  20%|██        | 10/50 [00:17<01:18,  1.96s/it, loss=nan, var=nan, cost=nan, err_std=nan, err_mean=+nan]

Epoch 10/50  loss=nan  var=nan  cost=nan  hedging_err=nan +/- nan


train:  30%|███       | 15/50 [00:27<01:08,  1.95s/it, loss=nan, var=nan, cost=nan, err_std=nan, err_mean=+nan]

Epoch 15/50  loss=nan  var=nan  cost=nan  hedging_err=nan +/- nan


train:  36%|███▌      | 18/50 [00:34<01:00,  1.89s/it, loss=nan, var=nan, cost=nan, err_std=nan, err_mean=+nan]


KeyboardInterrupt: 